# NCC Tutorial | Training Tempotron on MNIST
---
## Miranda Gonzales, PhD student 
---
#### Neuroscience Coding Club at UT San Antonio
#### 03 APRIL 2026

In [ ]:
# Load necessary packages

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from copy import deepcopy
import json


#### MNIST Inputs

In [ ]:
# Prep MNIST input arrays

# Load MNIST dataset from OpenML
mnist = fetch_openml('mnist_784', version=1)

# Convert the data into numpy arrays
x_mnist = mnist.data.to_numpy() # features (pixel values; grayscale)
y_mnist = mnist.target.to_numpy() # labels

# Convert data to the correct type
x_mnist = x_mnist.astype(np.float32)
y_mnist = y_mnist.astype(np.int64)

print(x_mnist.shape, x_mnist[0]) # shape is 70k samples, 784 pixels per sample
print(y_mnist.shape, y_mnist[0]) # shape is 70k samples

x_mnist_train = x_mnist[:50000]
y_mnist_train = y_mnist[:50000]
x_mnist_labelval = x_mnist[50000:60000]
y_mnist_labelval = y_mnist[50000:60000]
x_mnist_test = x_mnist[60000:]
y_mnist_test = y_mnist[60000:]

    

In [ ]:
# Plotting functions for MNIST inputs

def plot_mnist_trial(x,y):
    # Create the subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    
    # Plot the histogram on the first subplot
    x_reshape = x.reshape(28,28)
    ax1.imshow(x_reshape, cmap='gray') #hist(data, bins=30)
    ax1.set_title('Label: '+str(y))
    
    # Plot a line plot on the second subplot
    ax2.hist(x, bins=30)
    ax2.set_title('Histogram: Pixel values')
    ax2.set_xlabel('Pixel Value')
    ax2.set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

def plot_ttfs_raster(spike_times_2d, label):
    spike_times_flat = spike_times_2d.flatten()
    neuron_indices = np.arange(len(spike_times_flat))
    
    # Only keep neurons that spike
    valid = ~np.isinf(spike_times_flat)
    spike_times_flat = spike_times_flat[valid]
    total_active = int((len(spike_times_flat)/len(valid)) *100)
    neuron_indices = neuron_indices[valid]
    
    # Prepare data for eventplot: a list of spike time(s) per neuron
    events = [[] for _ in range(neuron_indices.max() + 1)]
    for idx, t in zip(neuron_indices, spike_times_flat):
        events[idx].append(t)
    
    # Plot raster
    plt.figure(figsize=(10, 6))
    plt.eventplot(events, colors='black', linelengths=1.5)
    plt.xlabel("Time (ms)")
    plt.ylabel("Pixel (flattened index)")
    plt.title("Raster Plot of TTFS-Encoded MNIST Input\nActive Neurons = "+str(total_active)+"% | Label = "+str(label)+" | Note: brighter pixels spike earlier")
    plt.tight_layout()
    plt.show()

In [ ]:
# Function to create spike times of input neurons from MNIST digits using Time-to-first spike (TTFS) scheme (plotting optional)

def input_spikes_mnist(x, y, t_min, t_max, threshold=0.0, if_plot=False):
    """
    Input parameters:
    x (np.array) : pixel values (grascale) 784 pixels per MNIST image
    y (int) : correct label assingment for MNIST image
    t_min : earliest possible spike time
    t_max : latest possible spike time
    threshold : pixel intensity below which no spike is generated (optional)
    if_plot (boolean; True or False) : will plot the spiking raster if True

    Returns:
    spike_times : 2D array of same shape as image, with spike times (inf if no spike)
    optional : 2 plots
    """
    
    norm_image = (x - x.min()) / (x.max() - x.min()) 
    spike_times = t_min + (1 - norm_image) * (t_max - t_min)
    spike_times[norm_image <= threshold] = np.inf  # no spike

    if if_plot == True:
        plot_mnist_trial(x, y)
        plot_ttfs_raster(spike_times, y)

    input_spikes = np.zeros((t_max, len(x)))
    for i in range(len(x)):
        t = spike_times[i]
        if t < t_max:
            if round(t)==t_max:
                offset=1
            else:
                offset=0
            input_spikes[round(t)-offset, i] = 1.0  # spike at first-available time
        
    return input_spikes 


In [ ]:
# Function to plot heatmap of MNIST pixel parameters (i.e., weights)

def plot_mnist_heatmap(weights_1d, title="Synaptic Weights to One Output Neuron", savefig=False, savename="weight_heatmap.pdf"):
    """
    Plots a 28x28 heatmap for weights from 784 input neurons to one output neuron.
    
    Parameters:
        weights_1d (array-like): A 1D array of 784 weights.
        title (str): Title of the plot.
    """
    assert len(weights_1d) == 784, "Input weight array must have 784 elements."
    
    weights_2d = np.reshape(weights_1d, (28, 28))
    
    plt.figure(figsize=(6, 6))
    sns.heatmap(weights_2d, cmap='viridis', cbar_kws={'label': 'Weight'}, square=True)
    plt.title(title)
    plt.axis('off')  # Optional: hide axis for cleaner image view
    plt.tight_layout()
    if savefig == True:
        plt.savefig(savename)
    plt.show()

In [ ]:
# Example code to create MNIST plots
ex_MNIST_spl = 0
spike_trains = input_spikes_mnist(x_mnist_train[ex_MNIST_spl], 
                                  y_mnist_train[ex_MNIST_spl], 
                                  t_min=0, t_max=30, 
                                  threshold=0.0, 
                                  if_plot=True)

#### Tempotron Code

In [ ]:
# Tau (membrane time constant) demonstration

# Relevant neuron model & simulation parameters
v0 = 2.12
dt = 1
time_per_state = 7
Tmax_padded = (time_per_state * 2)+110
t_padded = np.arange(0, Tmax_padded, dt)

# Example 1 of tau (to be plotted in black)
tau = 15 
tau_s = tau/4
kernelT_padded = v0 * (np.exp(-t_padded/tau) - np.exp(-t_padded/tau_s))

# Example 2 of tau (to be plotted in red)
tau_larger = 10 
tau_s_larger = tau_larger/4
kernelT_largertau = v0 * (np.exp(-t_padded/tau_larger) - np.exp(-t_padded/tau_s_larger))

# Add lines emulating voltage trace to the plot & show
plt.plot(t_padded, kernelT_padded, linestyle = '--', color = 'k')
plt.plot(t_padded, kernelT_largertau, linestyle = '-', color='r')
plt.plot(t_padded[10], kernelT_padded[10], marker ='.', color = 'k')
plt.plot(t_padded[10], kernelT_largertau[10], marker ='.', color = 'r')

plt.show()

In [ ]:
# Define Tempotron Neuron model as a pythonic class

class TempotronNeuron:
    def __init__(self, n_inputs, w_import=False, w_mean=0.001, w_stdev=0.0001, tau=15.0, tau_s=3.75, v0=2.12, v_thresh=1.0, lr=0.001):
        self.n_inputs = n_inputs
        self.tau = tau
        self.tau_s = tau_s
        self.v0 = v0
        self.v_thresh = v_thresh
        self.lr = lr
        self.reset_state()
        if np.isnan(w_import).all(): 
            self.weights = np.random.normal(w_mean, w_stdev, n_inputs)
        else:
            self.weights = w_import
            

    def reset_state(self):
        self.V_peak = 0.0
        self.V_trace = None

    def PSP_kernel(self, t_diff):
        """Post-synaptic potential kernel"""
        k = self.v0 * (np.exp(-t_diff / self.tau) - np.exp(-t_diff / self.tau_s))
        k[t_diff < 0] = 0
        return k

    def compute_voltage(self, spike_trains, time_window):
        all_K = np.zeros((self.n_inputs, len(time_window)))   
        for i in range(self.n_inputs):
            spikes = spike_trains[i]
            if np.sum(spikes) == 0:
                continue 
            t_diffs = np.convolve(spikes, time_window)[:len(time_window)] 
            all_K[i] = self.PSP_kernel(t_diffs)
        
        V = np.dot(self.weights, all_K)
        self.V_trace = V
        self.V_peak = np.max(V)
        return V

    def fire(self):
        return self.V_peak >= self.v_thresh

    def update_weights(self, spike_trains, time_window, target_output):
        pred = self.fire()
        if pred == target_output:
            return 0  # No error

        t_max = time_window[np.argmax(self.V_trace)]
        for i in range(self.n_inputs):
            spikes = spike_trains[i]
            if np.sum(spikes) == 0:
                continue
            t_diffs = t_max - np.array(np.where(spikes>0))
            valid_diffs = t_diffs[t_diffs >= 0] 
            if valid_diffs.size == 0:
                continue
            dV_dw = np.sum(self.PSP_kernel(valid_diffs))
            delta_w = self.lr * dV_dw
            if pred and not target_output:
                self.weights[i] -= delta_w
            elif not pred and target_output:
                self.weights[i] += delta_w
        return 1  # Error occurred

In [ ]:
# Define Tempotron Network layer as a pythonic class

class TempotronNetwork:
    def __init__(self, n_neurons, n_inputs, w_import_all, time_window, **neuron_params):
        if np.isnan(w_import_all).all():
            self.neurons = [TempotronNeuron(n_inputs, w_import=[np.nan], **neuron_params) for _ in range(n_neurons)]
        else:
            self.neurons = [TempotronNeuron(n_inputs, w_import=w_import_all[_], **neuron_params) for _ in range(n_neurons)]
        self.time_window = time_window
        self.error_log = []
        self.error_type_log = []
        self.true_acc_log = []
        self.all_targets = []
        for i in self.neurons:
            self.error_type_log.append([])
            self.true_acc_log.append([])
            self.all_targets.append([])

    def reset(self):
        for neuron in self.neurons:
            neuron.reset_state()

    def train_step(self, spike_trains, target_outputs):
        errors = []
        for neuron, target in zip(self.neurons, target_outputs):
            neuron.compute_voltage(spike_trains, self.time_window)
            error = neuron.update_weights(spike_trains, self.time_window, target)
            errors.append(error)
        self.error_log.append(np.sum(errors))
        for e, error in enumerate(errors):
            self.all_targets[e].append(target_outputs[e])
            self.error_type_log[e].append(error)
            self.true_acc_log[e].append(0)
        if np.sum(errors) == 0:
            if len(np.where(target_outputs > 0)) > 1:
                raise CustomError("Target output had more correct values than expected; see train_step function in TempotronNetwork.")
            self.true_acc_log[np.where(np.array(target_outputs) > 0)[0][0]][-1] = 1
        
        self.reset()

    def predict(self, spike_trains):
        preds = []
        for neuron in self.neurons:
            neuron.compute_voltage(spike_trains, self.time_window)
            preds.append(int(neuron.fire()))
        self.reset()
        return preds

#### Building & Running a Tempotron Network Simulation training on the MNIST dataset (TTFS scheme)

In [ ]:
# Training over pre-defined number of samples (option to train continuously across many epochs)

# update every time...will be used for plotting & saving data appropriately
date_info = "260403"

# Initialize basic parameters of interest
lr_combo = {'combo110': [110, 0.001]} # learning rate
w_m = 0.01                            # mean of weight initialization (from normal dist.)
w_sd = 0.001                          # standard deviation of weight initialization (from normal dist.)

# Define time window for each MNIST sample simulation
t_enc = 31 # ms
time_window = np.arange(0,t_enc,1)  

# Define number of epochs, total number of training samples, and how many often the network will be evaluated
# Note: 21 seconds to run 1000 spls w/ eval every 100 spls [MG running on MacBook Pro 2024]
epochs = 1 #2 # each epoch trains on all 50k training spls
n_train_spls = 1000 #50000*epochs # epoch max = 50000
n_eval_slice = 100 #2500

# Initialize dictionary to save final weights
dict_final_w = {}
dict_train_perform = {}

# For loop set-up to try out different learning rates associated with a unique experiment parameter-combo ID
for cx in lr_combo:
    param_combo = lr_combo[cx][0] 
    lr = lr_combo[cx][1] 
    param_id = 'w_m=0.01, w_sd=0.001, lr='+str(lr)

    # Add dictionary to save final weights for this loop's lr_combo
    dict_final_w['combo'+str(param_combo)] = {'Tempo0':[], 'Tempo1':[], 'Tempo2':[], 'Tempo3':[], 'Tempo4':[], 
                                              'Tempo5':[], 'Tempo6':[], 'Tempo7':[], 'Tempo8':[], 'Tempo9':[]}
    
    
    # Optional: Import previous final weights & reformat in np.array (10x784)
    #w_nparray = np.zeros((10,784))
    #for i in range(10):
    #    w_nparray[i] = np.array(dict_w_import_t2[cx]['Tempo'+str(i)])
    
    # Create a Tempotron network with 10 neurons and 784 input channels
    tempotron_net = TempotronNetwork(n_neurons=10, n_inputs=784, w_import_all=np.nan, w_mean=w_m, w_stdev=w_sd, time_window=time_window, lr=lr)
        # Note: to initialize weights from w_mean & w_stdev, use: w_import_all=np.nan
    

    # Initialize arrays to store evaluation metrics
    all_train_acc = np.zeros((((n_train_spls//n_eval_slice)+1)))
    all_total_slices = np.zeros((((n_train_spls//n_eval_slice)+1)))
    all_eval_slices = np.zeros((((n_train_spls//n_eval_slice)+1),10))
    x_lst_eval = np.zeros((((n_train_spls//n_eval_slice)+1),10))
    
    ###
    # Begin Training
    ###
    for n_eval_epoch in range(n_train_spls):
        # Resets sample ID back to 0 for every epoch
        n_eval = n_eval_epoch % 50000
        
        # Generate spike trains & target output for single MNIST digit
        spike_trains = input_spikes_mnist(x_mnist_train[n_eval], y_mnist_train[n_eval], t_min=0, t_max=t_enc, threshold=0.0, if_plot=False)
        spike_trains_T = spike_trains.T
        target_outputs = np.zeros(10)
        target_outputs[y_mnist_train[n_eval]] = 1
        
        # Train step
        tempotron_net.train_step(spike_trains_T, target_outputs)

        # Evaluation
        if (n_eval+1) % n_eval_slice == 0:
            x_lst_eval[(n_eval_epoch//n_eval_slice)+1][:] = n_eval_epoch
            all_train_acc[(n_eval_epoch//n_eval_slice)+1] = np.sum(tempotron_net.true_acc_log)/len(tempotron_net.true_acc_log[0])
            all_total_slices[(n_eval_epoch//n_eval_slice)+1] = np.sum([i[-n_eval_slice:] for i in tempotron_net.true_acc_log])/n_eval_slice
            
            rec_acc = np.sum([i[-n_eval_slice:] for i in tempotron_net.true_acc_log], axis=1)
            rec_tot = np.sum([i[-n_eval_slice:] for i in tempotron_net.all_targets], axis=1)
            all_eval_slices[(n_eval_epoch//n_eval_slice)+1] = rec_acc/rec_tot
    ###
    # End of Training
    ###
    
    # Save & plot accuracy vs training progression 
    dict_train_perform['combo'+str(param_combo)] = {'true_acc_log': tempotron_net.true_acc_log, 
                                                    'all_targets': tempotron_net.all_targets}

    # Plot & Save Accuracy across training
    plt.figure(figsize=(10, 6))
    plt_x = x_lst_eval.T
    plt.axhline(y=65, color='red', linestyle='--', linewidth=2, label='65%, 70%, 75%')
    plt.axhline(y=70, color='red', linestyle='--', linewidth=2)
    plt.axhline(y=75, color='red', linestyle='--', linewidth=2)
    plt.plot(plt_x[0], all_train_acc*100, color='black', linestyle=':', linewidth=2, label='All Tempotrons (Cumulative)')
    plt.plot(plt_x[0], all_total_slices*100, color='black', linewidth=2, label='All Tempotrons')
    for i in range(10):
        plt.plot(plt_x[i], all_eval_slices.T[i]*100, linewidth=0.75, label='Tempotron '+str(i))
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.5)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("MNIST training sample (1 epoch = 50k spls)")
    plt.title("Accuracy across Training | 2 full epochs\nCombo "+str(param_combo)+" | Param ID: "+param_id)
    plt.tight_layout()
    plt.savefig("TempotronMNIST_combo"+str(param_combo)+"_Acc_"+date_info+".pdf")
    plt.show()
    
    # save & plot final weights
    for n in range(10):
        dict_final_w['combo'+str(param_combo)]['Tempo'+str(n)] = list(tempotron_net.neurons[n].weights)

# Export all simulation final weight data to json output file
file_path = "TempotronMNIST_finW_"+date_info+".json"
with open(file_path, 'w') as json_file:
    json.dump(dict_final_w, json_file, indent=4)
print(f"Dictionary saved to {file_path}")


#### Accessing json file and making more figures to analyze final network state

In [ ]:
# Loading the dictionary for param ID 110
file_path = "TempotronMNIST_finW_"+date_info+".json"

with open(file_path, 'r') as json_file:
    loaded_dictionary = json.load(json_file)
print(f"Dictionary loaded from {file_path}:")

print(loaded_dictionary.keys())
print(loaded_dictionary['combo110'].keys())

In [ ]:
# Visualize: (1) spread of final weights per Tempotron digit class

# Loop through all the paramater-combo IDs
for combo_id in loaded_dictionary:
    # Loop through each neuron and plot weight as a scatter point; show plot when loop completes
    for n, neuro in enumerate(loaded_dictionary[trial]):
        x_lst = np.ones(784)*n
        plt.scatter(x_lst, loaded_dictionary[combo_id][neuro], s=0.1)
    plt.xticks([0,1,2,3,4,5,6,7,8,9])
    plt.title("Tempotron Final Weights | Hyperparam "+combo_id)
    plt.show()

    

In [ ]:
# Visualize: (2) heatmaps of weight matrix per Tempotron neuron

# Loop through all the paramater-combo IDs
for combo_id in loaded_dictionary:
    # Loop through all 10 Tempotron neurons and plot heatmap of weights with previously defined function
    for n, neuro in enumerate(loaded_dictionary[combo_id]):
        plot_mnist_heatmap(loaded_dictionary[combo_id][neuro], title="Weights | Tempotron "+str(n), savefig=False, savename="na.pdf")
        

### End of Tutorial
---
# Happy Coding! -MG